# 02 — Flash sale có thật sự lãi không? (kiểm định thống kê)

**Câu hỏi:** flash sale làm tăng doanh thu, nhưng sau khi trừ chiết khấu sâu
hơn và tỷ lệ huỷ cao hơn, nó có còn tạo ra lợi nhuận không?

Notebook này lặp lại và kiểm định thống kê chặt hơn cho kết luận đã có ở
`sql/03_phan_tich_2_flash_sale.sql` (Phân tích 2). Dùng `dim_order_customer`
/ `fact_order_item_customer` — đã loại 414 đơn vận hành nội bộ tạo ra để giữ
chỗ flash sale (`docs/data_quality.md` mục 14). Đây là lý do bắt buộc, không
phải tuỳ chọn: chính cụm đơn đó được tạo ra quanh flash sale, dùng bảng gốc
sẽ làm sai lệch đúng vào phép so sánh đang cần làm.

Lãi gộp dùng công thức đã chốt ở mục 15: `unitPrice - sellerDiscountTotal -
gia_nhap` (không trừ `platformDiscountTotal` vì Lazada hoàn riêng khoản đó).


In [1]:
import duckdb
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/clean/lazada.duckdb", read_only=True)

item_margin = con.execute("""
    SELECT
        f.orderItemId, f.product_code, f.is_completed,
        f.unitPrice, f.sellerDiscountTotal, f.platformDiscountTotal,
        p.gia_nhap,
        CASE WHEN f.campaign_flag IS NOT NULL THEN 'A_FS' ELSE 'B_Thuong' END AS nhom,
        (f.unitPrice - f.sellerDiscountTotal - p.gia_nhap) AS margin
    FROM fact_order_item_customer f
    JOIN dim_product p ON f.product_code = p.product_code
    WHERE p.gia_nhap IS NOT NULL
""").df()

print(f"{len(item_margin):,} dòng có giá vốn (loại 38/196 mã chưa ánh xạ được)")


21,053 dòng có giá vốn (loại 38/196 mã chưa ánh xạ được)


In [2]:
MIN_N = 20  # toi thieu 1 nhom de test co y nghia, khop nguong da dung o Phan tich 2

counts = item_margin.groupby(["product_code", "nhom"]).size().unstack(fill_value=0)
eligible = counts[(counts.get("A_FS", 0) >= MIN_N) & (counts.get("B_Thuong", 0) >= MIN_N)].index

print(f"Tổng số product_code có giá vốn: {counts.shape[0]}")
print(f"Product_code có mặt ở CẢ 2 nhóm với >= {MIN_N} dòng/nhóm: {len(eligible)}")

sub = item_margin[item_margin["product_code"].isin(eligible)].copy()
sub["margin_khoi_tao"] = np.where(sub["is_completed"], sub["margin"], 0)
print(f"Số dòng dùng để so sánh/kiểm định: {len(sub):,}")


Tổng số product_code có giá vốn: 158
Product_code có mặt ở CẢ 2 nhóm với >= 20 dòng/nhóm: 16
Số dòng dùng để so sánh/kiểm định: 12,566


## Bảng so sánh (tầng dòng — mỗi dòng là 1 lần mua 1 sản phẩm, không phải 1 đơn)

`AOV` ở đây dùng theo nghĩa "giá thực trả trung bình mỗi lượt mua"
(`unitPrice - sellerDiscountTotal - platformDiscountTotal`, đúng giá khách
trả sau khi trừ cả 2 loại discount) — khác với AOV tầng đơn dùng ở
`01_seasonality.ipynb` (`SUM(paidPrice)` theo `orderNumber`). Ở đây đang so
sánh giữa 2 nhóm SẢN PHẨM, nên hợp lý hơn khi đo ở tầng dòng.


In [3]:
def summarize(df):
    completed = df[df["is_completed"]]
    return pd.Series({
        "so_dong": len(df),
        "ty_le_huy_pct": 100 * (~df["is_completed"]).mean(),
        "AOV_gia_thuc_tra": (completed["unitPrice"] - completed["sellerDiscountTotal"] - completed["platformDiscountTotal"]).mean(),
        "do_sau_chiet_khau_pct": 100 * ((df["sellerDiscountTotal"] + df["platformDiscountTotal"]) / df["unitPrice"]).mean(),
        "lai_gop_TB_moi_don_hoan_tat": completed["margin"].mean(),
        "lai_gop_TB_moi_don_khoi_tao": df["margin_khoi_tao"].mean(),
        "lai_gop_trung_vi_moi_don_khoi_tao": df["margin_khoi_tao"].median(),
    })

summary = sub.groupby("nhom").apply(summarize, include_groups=False)
summary.index = summary.index.map({"A_FS": "A — Flash Sale", "B_Thuong": "B — Thường"})
summary.round(1)


,so_dong,ty_le_huy_pct,AOV_gia_thuc_tra,do_sau_chiet_khau_pct,lai_gop_TB_moi_don_hoan_tat,lai_gop_TB_moi_don_khoi_tao,lai_gop_trung_vi_moi_don_khoi_tao
nhom,,,,,,,
A — Flash Sale,"2,267.0",27.0,"82,007.0",10.2,"51,274.2","37,409.6","13,000.0"
B — Thường,"10,299.0",27.3,"49,627.3",6.4,"19,625.2","14,266.8","13,000.0"


## Kiểm định 1 — tỷ lệ huỷ: two-proportion z-test

Giả thuyết: H0 = tỷ lệ huỷ nhóm A (Flash Sale) bằng nhóm B (Thường). Đây là
so sánh 2 **tỷ lệ** (biến nhị phân: huỷ / không huỷ) trên mẫu đủ lớn (>2.000
và >10.000 dòng mỗi nhóm) — điều kiện chuẩn để dùng z-test thay vì test chính
xác kiểu Fisher (Fisher hợp hơn khi mẫu nhỏ).


In [4]:
A = sub[sub["nhom"] == "A_FS"]
B = sub[sub["nhom"] == "B_Thuong"]

count = [(~A["is_completed"]).sum(), (~B["is_completed"]).sum()]
nobs = [len(A), len(B)]
z_stat, p_val = proportions_ztest(count, nobs)
diff = count[0] / nobs[0] - count[1] / nobs[1]

print(f"Tỷ lệ huỷ A: {count[0]/nobs[0]:.1%}   Tỷ lệ huỷ B: {count[1]/nobs[1]:.1%}")
print(f"Chênh lệch (effect size, đơn vị điểm %): {diff*100:+.2f} điểm %")
print(f"z = {z_stat:.3f}   p-value = {p_val:.4f}")
print("=> KHÔNG bác bỏ H0 (p > 0.05)" if p_val > 0.05 else "=> Bác bỏ H0 (p <= 0.05)")


Tỷ lệ huỷ A: 27.0%   Tỷ lệ huỷ B: 27.3%
Chênh lệch (effect size, đơn vị điểm %): -0.26 điểm %
z = -0.255   p-value = 0.7987
=> KHÔNG bác bỏ H0 (p > 0.05)


## Kiểm định 2 — lãi gộp mỗi đơn khởi tạo: Mann-Whitney U (không dùng t-test)

**Vì sao Mann-Whitney U thay vì t-test:** `margin_khoi_tao` có một khối giá
trị đúng bằng **0** cho mọi đơn bị huỷ (~27% dòng ở cả 2 nhóm) — phân phối bị
dồn cục ở 0 rồi kéo dài đuôi phải sang các giá trị lãi dương, hoàn toàn không
đối xứng và không phải phân phối chuẩn. t-test giả định dữ liệu (hoặc trung
bình mẫu) xấp xỉ chuẩn — vi phạm giả định này với dữ liệu dồn cục/lệch mạnh
sẽ làm p-value không đáng tin. Mann-Whitney U so sánh **thứ hạng** thay vì
giá trị trung bình, không đòi hỏi phân phối chuẩn, phù hợp hơn hẳn với hình
dạng dữ liệu này.

**Effect size:** dùng hệ số tương quan hạng (rank-biserial correlation)
`r = 1 - 2U / (n1×n2)`, nằm trong [-1, 1] — quy ước đọc: |r| < 0.1 gần như
không đáng kể, 0.1–0.3 nhỏ, 0.3–0.5 vừa, > 0.5 lớn. **Đây là bước bắt buộc,
không phải tuỳ chọn** — với mẫu lớn (nghìn dòng), p-value gần như luôn có ý
nghĩa thống kê dù chênh lệch thực tế rất nhỏ; effect size mới cho biết chênh
lệch đó có *đáng để hành động* hay không.


In [5]:
u_stat, p_val2 = mannwhitneyu(A["margin_khoi_tao"], B["margin_khoi_tao"], alternative="two-sided")
n1, n2 = len(A), len(B)
r_effect = 1 - (2 * u_stat) / (n1 * n2)

print(f"Trung vị A: {A['margin_khoi_tao'].median():,.0f}đ   Trung vị B: {B['margin_khoi_tao'].median():,.0f}đ")
print(f"Trung bình A: {A['margin_khoi_tao'].mean():,.0f}đ   Trung bình B: {B['margin_khoi_tao'].mean():,.0f}đ")
print(f"U = {u_stat:,.0f}   p-value = {p_val2:.2e}")
print(f"Rank-biserial r = {r_effect:.4f}  ({'không đáng kể' if abs(r_effect) < 0.1 else 'nhỏ' if abs(r_effect) < 0.3 else 'vừa/lớn'})")


Trung vị A: 13,000đ   Trung vị B: 13,000đ
Trung bình A: 37,410đ   Trung bình B: 14,267đ
U = 12,394,578   p-value = 2.91e-06
Rank-biserial r = -0.0617  (không đáng kể)


## Kết luận

**Tỷ lệ huỷ:** chênh lệch -0,26 điểm % (A thấp hơn B rất nhẹ), p = 0,80 —
**không có ý nghĩa thống kê**. Bác bỏ hoàn toàn giả thuyết "flash sale huỷ
nhiều hơn hàng thường".

**Lãi gộp mỗi đơn khởi tạo:** p = 2,9×10⁻⁶ — có ý nghĩa thống kê rất mạnh,
nhưng **rank-biserial r ≈ -0,06 (không đáng kể)**. Đây là ví dụ kinh điển của
"significant nhưng không substantial": với mẫu hàng nghìn dòng, gần như bất
kỳ chênh lệch nào cũng cho p nhỏ, nhưng effect size mới nói lên chênh lệch đó
có đáng bận tâm không. Trung vị 2 nhóm **bằng nhau tuyệt đối (13.000đ)** —
đơn "điển hình" ở nhóm Flash Sale và nhóm Thường lãi như nhau. Chênh lệch lớn
ở **trung bình** (37.410đ so với 14.267đ) đến từ đuôi phân phối — một số ít
sản phẩm mẫu nhỏ có lãi/đơn rất cao trong nhóm Flash Sale (đã nêu ở Phân tích
2: SGLW, S5, S8, SW7...) kéo trung bình lên, không phản ánh đơn "điển hình".

**Khớp với kết luận đã có ở Phân tích 2** (dựa trên riêng SHRTT01 — sản phẩm
chủ lực, ít nhiễu nhất): flash sale **không** làm giảm lãi gộp so với bán
thường, và khác biệt (nếu có) không đủ lớn để coi là hiệu ứng thật ở cấp độ
"đơn điển hình" — chỉ đúng cho một nhóm nhỏ sản phẩm cụ thể, không phải hiệu
ứng chung của cơ chế flash sale.

## Giới hạn

Effect size nhỏ không có nghĩa là "không có nhóm sản phẩm nào flash sale
thực sự lãi hơn" — nó có nghĩa là **không có hiệu ứng đồng nhất trên toàn bộ
16 SKU**. Muốn biết sản phẩm nào thực sự khác biệt cần test riêng từng SKU
(giảm cỡ mẫu, giảm độ tin cậy) — không làm ở đây vì đã có bảng chi tiết theo
từng `product_code` ở Phân tích 2. z-test và Mann-Whitney giả định các dòng
độc lập với nhau — thực tế nhiều dòng cùng thuộc 1 đơn (dùng chung
`orderNumber`) nên không hoàn toàn độc lập; ảnh hưởng nhỏ vì phần lớn đơn chỉ
có 1–2 dòng, nhưng không phải zero.
